In [18]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
client = OpenAI()


In [19]:
response = client.embeddings.create(
    model = 'text-embedding-3-small',
    input=['Hello World', '안녕하세요']
)

response

CreateEmbeddingResponse(data=[Embedding(embedding=[0.004802703857421875, -0.054718017578125, 0.0455322265625, 0.031524658203125, -0.0283355712890625, -0.0296783447265625, -0.03143310546875, 0.0316162109375, -0.01421356201171875, 0.01534271240234375, 0.006114959716796875, -0.029693603515625, -0.0310516357421875, -0.0224456787109375, 0.0308380126953125, 0.0246429443359375, -0.056671142578125, 0.01373291015625, 0.0190277099609375, 0.0316162109375, 0.044952392578125, -0.0031719207763671875, -0.01451873779296875, -0.017120361328125, 0.0220947265625, -0.01282501220703125, -0.037841796875, 0.00812530517578125, 0.0139007568359375, -0.0462646484375, 0.0287017822265625, -0.048309326171875, -0.01175689697265625, 0.0189361572265625, -0.0011968612670898438, -0.0008835792541503906, 0.016876220703125, -0.0060577392578125, 0.01262664794921875, -0.021392822265625, 0.028533935546875, -0.025665283203125, 0.0013523101806640625, 0.050262451171875, -0.02813720703125, 0.031005859375, -0.07373046875, 0.026458

In [20]:
len(response.data[0].embedding)

1536

In [21]:
emb_vec0 = response.data[0].embedding
emb_vec1 = response.data[1].embedding

len(emb_vec0), len(emb_vec1)

(1536, 1536)

In [22]:
def text_to_embedding(texts, model='text-embedding-3-small'):
    texts = [text.replace('\n', ' ') for text in texts]
    response = client.embeddings.create(model=model, input=texts)

    return [data.embedding for data in response.data]

vecs = text_to_embedding(['Hello World', '안녕하세요'])

In [23]:
len(vecs[0]), len(vecs[1])

(1536, 1536)

In [24]:
import pandas as pd

review_df = pd.read_csv('fine_food_reviews_1k.csv', index_col=0)
print(review_df.info())
print(review_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Time       1000 non-null   int64
 1   ProductId  1000 non-null   str  
 2   UserId     1000 non-null   str  
 3   Score      1000 non-null   int64
 4   Summary    1000 non-null   str  
 5   Text       1000 non-null   str  
dtypes: int64(2), str(4)
memory usage: 450.7 KB
None
         Time   ProductId          UserId  Score  \
0  1351123200  B003XPF9BO  A3R7JR3FMEBXQB      5   
1  1351123200  B003JK537S  A3JBPC3WFUT5ZP      1   
2  1351123200  B000JMBE7M   AQX1N6A51QOKG      4   
3  1351123200  B004AHGBX4  A2UY46X0OSNVUQ      3   
4  1351123200  B001BORBHO  A1AFOYZ9HSM2CZ      5   

                                             Summary  \
0  where does one  start...and stop... with a tre...   
1                                  Arrived in pieces   
2          It isn't blanc mange, but isn't bad . . .   
3        The

In [25]:
review_df = review_df[['Summary', 'Text']]
review_df['Content'] = 'Title: ' + review_df['Summary'].str.strip() + '; Content: ' + review_df['Text'].str.strip()

In [26]:
review_df

,Summary,Text,Content
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title: where does one start...and stop... wit...
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title: Arrived in pieces; Content: Not pleased...
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"Title: It isn't blanc mange, but isn't bad . ...."
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,Title: These also have SALT and it's not sea s...
4,Happy with the product,My dog was suffering with itchy skin. He had ...,Title: Happy with the product; Content: My dog...
...,...,...,...
995,Delicious!,I have ordered these raisins multiple times. ...,Title: Delicious!; Content: I have ordered the...
996,Good Training Treat,My dog will come in from outside when I am tra...,Title: Good Training Treat; Content: My dog wi...
997,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...,Title: Jamica Me Crazy Coffee; Content: Wolfga...
998,Party Peanuts,Great product for the price. Mix with the Asia...,Title: Party Peanuts; Content: Great product f...


In [27]:
review_df['embedding'] = text_to_embedding(review_df['Content'].tolist())
review_df.head()

,Summary,Text,Content,embedding
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title: where does one start...and stop... wit...,"[0.036651611328125, -0.023193359375, -0.030471..."
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title: Arrived in pieces; Content: Not pleased...,"[0.01140594482421875, 0.0343017578125, -0.0411..."
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"Title: It isn't blanc mange, but isn't bad . ....","[0.0032253265380859375, 0.01265716552734375, -..."
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,Title: These also have SALT and it's not sea s...,"[-0.0028896331787109375, 0.01462554931640625, ..."
4,Happy with the product,My dog was suffering with itchy skin. He had ...,Title: Happy with the product; Content: My dog...,"[0.01204681396484375, -0.0560302734375, 0.0167..."


In [28]:
embed_df = review_df['embedding'].to_frame('embedding')
embed_df.index = review_df['Content']
embed_df

,embedding
Content,
Title: where does one start...and stop... with a treat like this; Content: Wanted to save some to bring to my Chicago family but my North Carolina family ate all 4 boxes before I could pack. These are excellent...could serve to anyone,"[0.036651611328125, -0.023193359375, -0.030471..."
"Title: Arrived in pieces; Content: Not pleased at all. When I opened the box, most of the rings were broken in pieces. A total waste of money.","[0.01140594482421875, 0.0343017578125, -0.0411..."
"Title: It isn't blanc mange, but isn't bad . . .; Content: I'm not sure that custard is really custard without eggs. But this comes close. I got it for use in a ""Vegan pancake"" recipe. We were having houseguests who were Vegan and I wanted to make some special breakfasts while they were here. One of the cooking/recipe sites had a recipe using this and there were lots of great reviews. I tried the recipe and it turned out like wallpaper paste -- yuck!<br />However, the so-called custard isn't so bad. I think it's probably just cornstarch and annatto (yellow coloring with a slight flavor). It's fun playing with it. You could dress it up with fruit. Seems to come out on the thin side when you make it as directed, so I use less milk because I like my custards to set firm. As a custard sauce it's fine. I would say it tastes something between a pudding and a custard.<br /><br />If you want a really good egg-free ""custard"" get an original recipe for ""blanc mange."" It takes a lot longer to make, but it's certainly worth the difference.","[0.0032253265380859375, 0.01265716552734375, -..."
"Title: These also have SALT and it's not sea salt.; Content: I like the fact that you can see what you're getting and that there are no bones or dark meat. There are 7 nice big chunks in every jar.<br /><br />These taste like tuna in a can but, because they're preserved in glass, you don't have to worry about either aluminum or BPA; BUT ... they are not just tuna and spring water.<br /><br />There is salt in there, too, and it's not healthy sea salt, it's toxic table salt.<br /><br />I am trying to contact Tonnino to confirm that. I might be wrong because the label states that the ingredients are ""tuna fish"" but the sticker on the top clarifies that it is the smaller (healthier) yellowfin, so the ""salt"" listed in the ingredients might be sea salt but, if it was, why don't they say so?<br /><br />Without confirmation, I will continue to look for a salt-free olive-oil free tuna preserved in glass.<br /><br />If you know of one, please contact me!","[-0.0028896331787109375, 0.01462554931640625, ..."
Title: Happy with the product; Content: My dog was suffering with itchy skin. He had been eating Natural Choice brand (cheaper) since he was a puppy. I was nervous to change foods. The vet suggested to change foods sand see if the skin issues cleared up. Wellness brand did the job. My dog seems to love the food and the skin issues cleared up within a few weeks.,"[0.01204681396484375, -0.0560302734375, 0.0167..."
...,...
Title: Delicious!; Content: I have ordered these raisins multiple times. They are always great and arrive timely. I can't go back to store bought chocolate covered raisins now! Love this product.,"[0.016448974609375, -0.039154052734375, -0.026..."
Title: Good Training Treat; Content: My dog will come in from outside when I am training her and look at the cupboard waiting for her treat. When I use the clicker training method she comes because she knows she has something special.,"[-0.0230255126953125, -0.0139007568359375, 0.0..."
Title: Jamica Me Crazy Coffee; Content: Wolfgang Puck's Jamaica Me Crazy is that wonderful blend of island flavors in a coffee. Have loved it from the first time tasting. Great product.,"[-0.029632568359375, -0.045745849609375, -0.02..."


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

def review_vector_search(query, embed_df=embed_df, top_n=5):
    query_emb = text_to_embedding([query])

    df = embed_df.copy()
    df['cos_sim'] = df['embedding'].apply(lambda emb: cosine_similarity([emb], query_emb)[0, 0])

    df = df.sort_values('cos_sim', ascending=False).head()
    df = df.reset_index()
    df = df[['Content', 'cos_sim']]

    return df

pd.set_option('display.max_colwidth', None)
review_vector_search('delicious fruit')

,Content,cos_sim
0,"Title: Delicious!; Content: For anyone who says ""I don't like fruitcake"" or anyone who's never had fruitcake and wonders what all the fuss is about, try this. (As long as you're not allergic to tree nuts or any other ingredient.) It's chock-a-block with nuts and moist fruit. I will definitely be buying more.",0.649448
1,"Title: Delicious .; Content: These plums are sweet and juicy, and the aroma is like perfume. And it doesn't hurt that they are good for you, too.",0.612282
2,"Title: Delicious!; Content: Wonderful! Deep, rich, pure black raspberry syrup! Absolutely delicious on waffles, cheesecake, ice cream, yogurt, drinks, etc. Thrilled to see that there are at least some berry syrup makers who do not feel the need to ""sour"" the flavor of perfect berry products with citric acid!",0.599917
3,"Title: These are Delicious!; Content: Great taste, right price, fabulous snack! It is the only fruit I can get my little one to eat and I can't keep my high school son out of them either. They are great pick-me-ups on the way to my daughter's soccer practice or before my early morning run. And best of all, Amazon ships these right to my door every month. No more finding the right store who carries them. I just set up the automatic recurring shipment once and it works like a charm!",0.591969
4,Title: Delicious!; Content: I have ordered these raisins multiple times. They are always great and arrive timely. I can't go back to store bought chocolate covered raisins now! Love this product.,0.502419


In [30]:
review_vector_search('best coffee')

,Content,cos_sim
0,"Title: Best coffee ever!; Content: In my opinion this is the best coffee ever! I've been drinking coffee for 50 plus years and this is what I serve to myself and friends. However, I wish I could find this grind in a pound size, so I could make a full pot rather than just a single cup.",0.612534
1,"Title: Great Coffee; Content: I have a coffee maker that grinds my coffee beans. It's hard to find whole bean decafinated coffee. When I find it in the brand that I like, I am excited. Seattle's Best is my favorite.",0.610797
2,"Title: Better than you-know-who's coffee...; Content: So my wife is a latte freak, and nursing, so decaf is the approved type. After the Senseo left the market, I struggled and found the <a href=""http://www.amazon.com/gp/product/B0047BIWSK"">Aerobie AeroPress Coffee and Espresso Maker</a> which is like a French Press for the 21st century. After getting our recipe figured out, my wife, who's been buying Venti Decaf Latte's at $4 a pop almost daily for years now declares that Seattle's best Level 3 Decaf in her home-made Latte is the best coffee she can get. We've tried other bands, and this is her favorite, hands down!",0.605585
3,"Title: BEST cup of coffee I've ever had!; Content: I thought I'd splurge and try this coffee. It costs much more than other decaf K-Cup options. But I hoped that meant it was better coffee. It IS better coffee. I've never had a better cup of coffee than this. It is excellent when compared to any other decaf or regular coffee I've tried.<br /><br />If you like BOLD, FLAVORFUL decaf coffee try this coffee and you'll really like it.",0.603422
4,"Title: BEST cup of coffee I've ever had!; Content: I thought I'd splurge and try this coffee. It costs much more than other decaf K-Cup options. But I hoped that meant it was better coffee. It IS better coffee. I've never had a better cup of coffee than this. It is excellent when compared to any other decaf or regular coffee I've tried.<br /><br />If you like BOLD, FLAVORFUL decaf coffee try this coffee and you'll really like it.",0.603387
